# ArtAug image-to-image on Colab T4 (low memory)

This notebook uses the official `ECNU-CILab/ArtAug-lora-FLUX.1dev-v1` adapter with DiffSynth-Studio and processes the 30 existing images in `data/artaug_input/`. It saves outputs only under `data/artaug_generated/` and adds manifest rows with `approval_status=pending`. It never changes the original dataset, split, training code, classifier checkpoints, evaluation results, or Streamlit app.

## Important facts and safeguards

- The ArtAug model card describes an aesthetics-enhancement LoRA for FLUX.1-dev. The published ModelScope model metadata lists `txt2img` as its task; it does not promise hand-preserving image editing.
- DiffSynth-Studio's official `FluxImagePipeline` does expose `input_image` and `denoising_strength`. This makes image-to-image technically callable with the base pipeline and the FLUX LoRA, but it does **not** guarantee the gesture, fingers, or composition will be preserved. The notebook starts with one image per class at low denoising strength and requires manual review before processing the remaining 27.
- The official LoRA file uses legacy keys such as `lora_unet_double_blocks_0_img_attn_proj.lora_down.weight`. Diffusers' generic `load_lora_weights()` may reject this format. This notebook uses DiffSynth's FLUX-specific `FluxLoRALoader` through `pipe.load_lora(...)` and verifies that quantized transformer modules received LoRA weights.
- FLUX Transformer and T5 are required to use 4-bit NF4. There is no full-precision or FP16 fallback for either. CLIP and VAE use FP16 with DiffSynth CPU offload. T4 computation uses FP16, not the official low-VRAM example's BF16 compute setting.
- Model weights are stored persistently in Google Drive under `/content/drive/MyDrive/ArtAug_Models/FLUX.1-dev/` and `/content/drive/MyDrive/ArtAug_Models/ArtAug-lora-FLUX.1dev-v1/`. The Hugging Face Hub downloader verifies remote file sizes, skips complete files, and downloads only missing or incomplete files directly to Drive. Re-running after interruption repeats the check and retries incomplete downloads. Model weights are not stored solely in `/root/.cache`.
- Authentication uses the Colab Secret `HF_TOKEN`; the token is not displayed or written to the notebook.
- DiffSynth 2.2.0 online NF4 conversion loads the full FP checkpoint state dict before quantizing. The notebook checks available RAM against a conservative budget and stops before loading if insufficient. Transformer and T5 are never silently changed to FP16.
- FLUX.1-dev access and its non-commercial license conditions apply. Generated media remains pending until you inspect it; do not claim improved classifier accuracy from generation alone.

Before running, put `Assignment_2/data/` in Google Drive and add a Colab Secret named `HF_TOKEN`, granting this notebook access. Select a Colab T4 runtime. Do not copy the full original dataset or any validation/test images into this workflow.

## 1. Environment and GPU check

This cell only inspects the runtime. It does not install packages or access model weights.

In [ ]:
import importlib.metadata as metadata
import importlib.util
import os
import platform
import shutil
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
for distribution in ("DiffSynth", "diffusers", "transformers", "bitsandbytes", "safetensors", "torch"):
    try:
        print(f"{distribution}: {metadata.version(distribution)}")
    except metadata.PackageNotFoundError:
        print(f"{distribution}: not installed")

try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"System RAM: {ram.total / 1024**3:.2f} GiB total; {ram.available / 1024**3:.2f} GiB available")
except ImportError:
    print("System RAM: psutil unavailable; this optional probe was skipped.")

if shutil.which("nvidia-smi"):
    os.system("nvidia-smi")

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is unavailable. Select Runtime > Change runtime type > T4 GPU, then rerun.")
gpu = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"CUDA GPU: {gpu}; compute capability {capability[0]}.{capability[1]}")
print(f"VRAM: {total_bytes / 1024**3:.2f} GiB total; {free_bytes / 1024**3:.2f} GiB free")
if "T4" not in gpu:
    print("WARNING: this is not a T4; results will not verify the target T4 configuration.")
if capability < (7, 5):
    raise RuntimeError("This notebook expects a T4-class GPU (compute capability 7.5) or newer.")

## 2. Locate the project data and cached model files

Mount Drive and point `DATA_DIR` at the copied Assignment 2 `data` folder. FLUX and ArtAug weights are stored separately under `/content/drive/MyDrive/ArtAug_Models/`. The download cell uses Hugging Face metadata to validate every selected file, skips complete files, and saves source repository paths directly into those Drive folders.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
default_data_dir = Path("/content/drive/MyDrive/Hand_Gesture_Project/Assignment_2/data")
entered = input(f"Path to Assignment_2/data on Drive [{default_data_dir}]: ").strip()
DATA_DIR = Path(entered).expanduser() if entered else default_data_dir
CLASSES = ("peace", "stop", "thumbs_up")
INPUT_DIR = DATA_DIR / "artaug_input"
GENERATED_DIR = DATA_DIR / "artaug_generated"
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}. Copy Assignment_2/data to Drive first.")
for class_name in CLASSES:
    class_input = INPUT_DIR / class_name
    class_output = GENERATED_DIR / class_name
    if not class_input.is_dir() or not class_output.is_dir():
        raise FileNotFoundError(f"Missing input or generated class directory for {class_name} under {DATA_DIR}.")
    files = sorted(p for p in class_input.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    if len(files) != 10:
        raise ValueError(f"Expected 10 input images in {class_input}; found {len(files)}.")
    print(f"{class_name}: {len(files)} input images; generated directory exists")
print("Project data check passed:", DATA_DIR)

In [ ]:
import os
from pathlib import Path

BASE_MODEL_ID = "black-forest-labs/FLUX.1-dev"
LORA_REPO_ID = "ECNU-CILab/ArtAug-lora-FLUX.1dev-v1"
HF_MODEL_CACHE_NAME = "models--black-forest-labs--FLUX.1-dev"
HF_LORA_CACHE_NAME = "models--ECNU-CILab--ArtAug-lora-FLUX.1dev-v1"

def diffusers_flux_files(directory):
    directory = Path(directory)
    transformer = sorted((directory / "transformer").glob("diffusion_pytorch_model-*-of-00003.safetensors"))
    t5 = sorted((directory / "text_encoder_2").glob("model-*-of-00002.safetensors"))
    clip = directory / "text_encoder" / "model.safetensors"
    vae = directory / "vae" / "diffusion_pytorch_model.safetensors"
    tokenizers = (directory / "tokenizer", directory / "tokenizer_2")
    expected_transformer = {f"diffusion_pytorch_model-{i:05d}-of-00003.safetensors" for i in range(1, 4)}
    expected_t5 = {f"model-{i:05d}-of-00002.safetensors" for i in range(1, 3)}
    if (
        {path.name for path in transformer} == expected_transformer
        and {path.name for path in t5} == expected_t5
        and clip.is_file()
        and vae.is_file()
        and all(path.is_dir() for path in tokenizers)
    ):
        return {
            "transformer": transformer,
            "clip": clip,
            "t5": t5,
            "vae": vae,
            "tokenizer_1": tokenizers[0],
            "tokenizer_2": tokenizers[1],
        }
    return None

def hf_hub_cache_roots():
    roots = []
    if os.environ.get("HF_HUB_CACHE"):
        roots.append(Path(os.environ["HF_HUB_CACHE"]))
    if os.environ.get("HF_HOME"):
        roots.append(Path(os.environ["HF_HOME"]) / "hub")
    roots.append(Path.home() / ".cache" / "huggingface" / "hub")
    return list(dict.fromkeys(path.expanduser() for path in roots))

direct_candidates = []
for env_name in ("DIFFSYNTH_MODEL_BASE_PATH",):
    if os.environ.get(env_name):
        root = Path(os.environ[env_name]).expanduser()
        direct_candidates.extend((root / BASE_MODEL_ID, root))
direct_candidates.extend((Path("/content/models") / BASE_MODEL_ID, Path.cwd() / "models" / BASE_MODEL_ID))
snapshot_candidates = []
for cache_root in hf_hub_cache_roots():
    snapshot_candidates.extend((cache_root / HF_MODEL_CACHE_NAME / "snapshots").glob("*"))
MODEL_DIR = next((path.resolve() for path in direct_candidates + snapshot_candidates if diffusers_flux_files(path)), None)
if MODEL_DIR is None:
    raise FileNotFoundError(
        "Could not find the complete cached Diffusers FLUX layout (three Transformer shards, CLIP, two T5 shards, VAE, tokenizers). "
        "No download was attempted. Check the existing Hugging Face cache path."
    )
MODEL_FILES = diffusers_flux_files(MODEL_DIR)

lora_candidates = [
    Path("/content/models/lora/merged_lora.safetensors"),
    Path("/content/models") / LORA_REPO_ID / "merged_lora.safetensors",
]
if os.environ.get("DIFFSYNTH_MODEL_BASE_PATH"):
    lora_root = Path(os.environ["DIFFSYNTH_MODEL_BASE_PATH"]).expanduser()
    lora_candidates.append(lora_root / LORA_REPO_ID / "merged_lora.safetensors")
for cache_root in hf_hub_cache_roots():
    lora_candidates.extend((cache_root / HF_LORA_CACHE_NAME / "snapshots").glob("*/merged_lora.safetensors"))
ART_AUG_PATH = next((path for path in lora_candidates if path.is_file()), None)
if ART_AUG_PATH is None:
    raise FileNotFoundError("Cached ArtAug merged_lora.safetensors was not found. No download was attempted.")

print("FLUX Diffusers cache:", MODEL_DIR)
for component, paths in MODEL_FILES.items():
    if isinstance(paths, list):
        print(f"{component}: {len(paths)} local shard(s):", [path.name for path in paths])
    else:
        print(f"{component}:", paths)
print("ArtAug LoRA cache:", ART_AUG_PATH)
print("Cache discovery complete; no model download or symlink creation was requested.")

## 3. Install only missing inference packages

Known-good runtime versions from the prior component-loading session are DiffSynth 2.2.0, Diffusers 0.42.1, Transformers 5.22.1, bitsandbytes 0.50.2, Python 3.13. The Diffusers pipeline is deliberately not used. Only missing required packages are installed, pinned to the known runtime versions, with dependency changes disabled. Existing versions are never upgraded automatically; unexpected versions stop the notebook for review.

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

required = {
    "diffsynth": ("DiffSynth", "2.2.0"),
    "bitsandbytes": ("bitsandbytes", "0.50.2"),
    "safetensors": ("safetensors", None),
    "transformers": ("transformers", "5.22.1"),
}
for module_name, (distribution, expected_version) in required.items():
    if importlib.util.find_spec(module_name) is None:
        spec = f"{distribution}=={expected_version}" if expected_version else distribution
        print(f"Installing missing package only: {spec} (without dependency upgrades)")
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", spec], check=True)

mismatches = []
for module_name, (distribution, expected_version) in required.items():
    actual = metadata.version(distribution)
    print(f"{distribution}: {actual}")
    if expected_version and actual != expected_version:
        mismatches.append(f"{distribution}: expected {expected_version}, found {actual}")
if mismatches:
    raise RuntimeError("Dependency versions differ from the previously tested Colab environment. No packages were upgraded automatically. Review before proceeding:\n" + "\n".join(mismatches))

from safetensors import safe_open
import bitsandbytes as bnb
import diffsynth
print("Required imports are available; Diffusers is not used to load or convert the ArtAug adapter.")

## 4. Configure separate model components

The Diffusers Transformer and T5 shard lists are passed as `ModelConfig(path=list[str])`; DiffSynth 2.2.0's `ModelConfig.path` and safetensors loader explicitly accept lists. The matching Diffusers FLUX state-dict converter is part of DiffSynth's model config. Transformer and T5 retain explicit bitsandbytes NF4. CLIP and VAE use FP16 with CPU offload. Online NF4 conversion is RAM-guarded below; on the reported 12.7 GiB Colab runtime the notebook is expected to stop before allocation unless a compatible NF4 pipeline is already loaded in memory.

In [ ]:
import os
import torch
from diffsynth.core.quant import QuantizeConfig
from diffsynth.pipelines.flux_image import FluxImagePipeline, ModelConfig
from diffsynth.version import __version__ as DIFFSYNTH_VERSION

if DIFFSYNTH_VERSION != "2.2.0":
    raise RuntimeError(f"This notebook's audited loader behavior targets DiffSynth 2.2.0; found {DIFFSYNTH_VERSION}.")
if torch.cuda.get_device_capability(0) < (7, 5):
    raise RuntimeError("The configured FP16 compute path requires a T4-class GPU or newer.")

NF4 = QuantizeConfig(method="bitsandbytes_nf4")
FP16_CPU_OFFLOAD = {
    "offload_dtype": torch.float16,
    "offload_device": "cpu",
    "onload_dtype": torch.float16,
    "onload_device": "cuda",
    "preparing_dtype": torch.float16,
    "preparing_device": "cuda",
    "computation_dtype": torch.float16,
    "computation_device": "cuda",
}
os.environ["DIFFSYNTH_SKIP_DOWNLOAD"] = "True"

TRANSFORMER_PATHS = [str(path) for path in MODEL_FILES["transformer"]]
T5_PATHS = [str(path) for path in MODEL_FILES["t5"]]
MODEL_CONFIGS = [
    ModelConfig(path=TRANSFORMER_PATHS, quantize=NF4),
    ModelConfig(path=str(MODEL_FILES["clip"]), **FP16_CPU_OFFLOAD),
    ModelConfig(path=T5_PATHS, quantize=NF4),
    ModelConfig(path=str(MODEL_FILES["vae"]), **FP16_CPU_OFFLOAD),
]
TOKENIZER_1_CONFIG = ModelConfig(path=str(MODEL_FILES["tokenizer_1"]))
TOKENIZER_2_CONFIG = ModelConfig(path=str(MODEL_FILES["tokenizer_2"]))

if not isinstance(MODEL_CONFIGS[0].path, list) or not isinstance(MODEL_CONFIGS[2].path, list):
    raise TypeError("DiffSynth shard configs must preserve list[str] paths.")
print("DiffSynth:", DIFFSYNTH_VERSION)
print("Transformer shards:", TRANSFORMER_PATHS)
print("T5 shards:", T5_PATHS)
print("NF4 quantization configured for Transformer and T5; CLIP/VAE use FP16 CPU offload.")
print("DiffSynth downloads disabled:", os.environ["DIFFSYNTH_SKIP_DOWNLOAD"])

## 5. Assemble the inference pipeline

This uses DiffSynth's FLUX pipeline assembly from separately configured cached components. It does not call `FluxPipeline.from_pretrained()` and does not create a full-precision copy of the pipeline. If a compatible DiffSynth pipeline with verified NF4 modules is already present in the current kernel, it is reused.

In [ ]:
def nf4_inventory(module):
    params4bit = sum(1 for parameter in module.parameters() if parameter.__class__.__name__ == "Params4bit")
    layers4bit = sum(1 for layer in module.modules() if layer.__class__.__name__ == "Linear4bit")
    return {"Params4bit": params4bit, "Linear4bit": layers4bit}

def has_nf4(module):
    info = nf4_inventory(module)
    return info["Params4bit"] > 0 or info["Linear4bit"] > 0

existing_pipe = globals().get("pipe")
reuse_existing = (
    isinstance(existing_pipe, FluxImagePipeline)
    and getattr(existing_pipe, "dit", None) is not None
    and getattr(existing_pipe, "text_encoder_2", None) is not None
    and has_nf4(existing_pipe.dit)
    and has_nf4(existing_pipe.text_encoder_2)
)

if reuse_existing:
    pipe = existing_pipe
    print("Reusing the already-loaded DiffSynth pipeline and its NF4 components.")
else:
    try:
        import psutil
    except ImportError as exc:
        raise RuntimeError("psutil is required for the host-RAM safety check; model loading is blocked, no package was installed.") from exc
    largest_fp_component_bytes = max(
        sum(path.stat().st_size for path in MODEL_FILES["transformer"]),
        sum(path.stat().st_size for path in MODEL_FILES["t5"]),
    )
    required_ram_bytes = int(largest_fp_component_bytes * 1.25 + 2 * 1024**3)
    available_ram_bytes = psutil.virtual_memory().available
    print(f"Largest FP checkpoint component: {largest_fp_component_bytes / 1024**3:.2f} GiB")
    print(f"Available host RAM: {available_ram_bytes / 1024**3:.2f} GiB; conservative load budget: {required_ram_bytes / 1024**3:.2f} GiB")
    if available_ram_bytes < required_ram_bytes:
        raise MemoryError(
            "Refusing to start online NF4 conversion. DiffSynth 2.2.0's load_model path materializes the complete FP state dict before quantize_model; "
            "the available RAM is below the conservative budget. Reuse an already-loaded NF4 pipeline or provide an officially serialized pre-quantized NF4 checkpoint. "
            "The Diffusers FP shards are not themselves NF4; no weights were loaded and no download was attempted."
        )
    torch.cuda.empty_cache()
    try:
        pipe = FluxImagePipeline.from_pretrained(
            torch_dtype=torch.float16,
            device="cuda",
            model_configs=MODEL_CONFIGS,
            tokenizer_1_config=TOKENIZER_1_CONFIG,
            tokenizer_2_config=TOKENIZER_2_CONFIG,
            vram_limit=max(1.0, torch.cuda.mem_get_info()[0] / (1024 ** 3) - 0.5),
        )
    except Exception as exc:
        raise RuntimeError(
            "DiffSynth failed to assemble the local Diffusers shards. No model download was allowed. "
            "Do not retry with unquantized Transformer/T5 or remove the RAM guard."
        ) from exc

transformer_nf4 = nf4_inventory(pipe.dit)
t5_nf4 = nf4_inventory(pipe.text_encoder_2)
print("Transformer 4-bit inventory:", transformer_nf4)
print("T5 4-bit inventory:", t5_nf4)
if not has_nf4(pipe.dit) or not has_nf4(pipe.text_encoder_2):
    raise RuntimeError("NF4 verification failed for Transformer or T5. Generation is blocked; no precision fallback is permitted.")
print("Verified NF4 is present in both required components.")
print("DiffSynth dynamic VRAM management:", getattr(pipe, "vram_management_enabled", False))
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"VRAM after assembly: {free_bytes / 1024**3:.2f} GiB free / {total_bytes / 1024**3:.2f} GiB total")

## 6. Inspect and load the official ArtAug LoRA

The safetensors header is inspected without reading all LoRA tensors into RAM. The file must contain the expected legacy FLUX keys. DiffSynth's own `pipe.load_lora(pipe.dit, ...)` invokes its `FluxLoRALoader` conversion; there is no hand-written or speculative key conversion.

In [ ]:
with safe_open(str(ART_AUG_PATH), framework="pt", device="cpu") as lora_file:
    lora_keys = list(lora_file.keys())
legacy_keys = [key for key in lora_keys if key.startswith("lora_unet_")]
down_keys = [key for key in legacy_keys if key.endswith(".lora_down.weight")]
up_keys = [key for key in legacy_keys if key.endswith(".lora_up.weight")]
print(f"LoRA header: {len(lora_keys)} tensors; {len(legacy_keys)} legacy FLUX keys; {len(down_keys)} down / {len(up_keys)} up matrices")
print("Example keys:", legacy_keys[:3])
if not legacy_keys or not down_keys or len(down_keys) != len(up_keys):
    raise ValueError("LoRA header is not the expected complete legacy FLUX format. Adapter loading is blocked.")

try:
    if hasattr(pipe, "clear_lora"):
        pipe.clear_lora(verbose=0)
    pipe.load_lora(pipe.dit, str(ART_AUG_PATH), alpha=1.0, hotload=True, verbose=1)
except Exception as exc:
    raise RuntimeError(
        "DiffSynth could not convert/apply the official ArtAug LoRA to the NF4 FLUX transformer. "
        "Do not fall back to Diffusers load_lora_weights() or silently continue."
    ) from exc

patched_modules = []
patched_pairs = 0
for name, module in pipe.dit.named_modules():
    lora_a = getattr(module, "lora_A_weights", [])
    lora_b = getattr(module, "lora_B_weights", [])
    if lora_a or lora_b:
        if len(lora_a) != len(lora_b):
            raise RuntimeError(f"Unbalanced LoRA hotload weights on transformer module {name!r}.")
        patched_modules.append(name)
        patched_pairs += len(lora_a)
print(f"Verified LoRA hotload: {patched_pairs} A/B weight pair(s) across {len(patched_modules)} transformer modules.")
if not getattr(pipe.dit, "vram_management_enabled", False):
    raise RuntimeError("DiffSynth did not enable the VRAM-managed hotload mode required by quantized NF4 weights.")
if not patched_modules or patched_pairs == 0:
    raise RuntimeError("LoRA hotload added no A/B pairs to transformer modules; generation is blocked.")
if patched_pairs != len(down_keys):
    raise RuntimeError(f"Only {patched_pairs}/{len(down_keys)} expected LoRA pairs were hot-loaded; generation is blocked.")
print("ArtAug adapter is loaded and verified.")

## 7. Verify image-to-image call support

The official DiffSynth `FluxImagePipeline` API accepts `input_image` and `denoising_strength`; this is the interface used below. This confirms callable img2img support in the base pipeline, not that the ArtAug authors validated gesture-preserving img2img. Low denoising strength reduces the edit but cannot guarantee the original fingers or hand remain unchanged.

In [ ]:
import inspect
signature = inspect.signature(FluxImagePipeline.__call__)
supported = {"input_image", "denoising_strength"}.issubset(signature.parameters)
print("FluxImagePipeline.__call__ parameters:", ", ".join(name for name in ("input_image", "denoising_strength", "prompt", "seed", "num_inference_steps") if name in signature.parameters))
required_parameters = {"input_image", "denoising_strength", "seed", "num_inference_steps"}
if not required_parameters.issubset(signature.parameters):
    raise RuntimeError(f"Installed DiffSynth API is missing required img2img/seed parameters: {sorted(required_parameters - set(signature.parameters))}")
if not supported:
    raise RuntimeError("Installed DiffSynth API does not expose the verified img2img parameters; no images will be generated.")
print("Image-to-image, integer seed, and step parameters are present in the installed API.")
print("DiffSynth 2.2.0 FluxImagePipeline.__call__ decodes and returns one PIL Image; the generator below saves that image directly.")

## 8. Generate a small test batch (3 images)

One selected original per class is processed first. Inputs are resized proportionally so the longest side is at most 768 pixels and dimensions are multiples of 64. Outputs use deterministic seeds and filenames. Existing outputs are not overwritten. If this test fails or damages the gesture, stop here; do not run the remaining batch.

In [ ]:
import csv
import hashlib
import math
import os
from pathlib import Path
from PIL import Image, ImageOps

TEST_INPUTS = {
    "peace": "IMG_2351.jpeg",
    "stop": "IMG_2443.jpeg",
    "thumbs_up": "IMG_2546.jpeg",
}
DENOISING_STRENGTH = 0.15
INFERENCE_STEPS = 12
MAX_IMAGE_SIDE = 768
PROMPTS = {
    "peace": "Enhance this photograph gently. Preserve the same peace hand gesture, the existing two raised fingers and all finger positions, the visible hand, viewpoint, framing, and background. Improve only natural photo clarity and lighting. Do not add, remove, or reshape fingers or hands.",
    "stop": "Enhance this photograph gently. Preserve the same stop gesture, open palm, all five extended fingers and their positions, the visible hand, viewpoint, framing, and background. Improve only natural photo clarity and lighting. Do not add, remove, or reshape fingers or hands.",
    "thumbs_up": "Enhance this photograph gently. Preserve the same thumbs-up gesture, upward thumb, curled fingers and their positions, the visible hand, viewpoint, framing, and background. Improve only natural photo clarity and lighting. Do not add, remove, or reshape fingers or hands.",
}
NEGATIVE_PROMPT = "extra fingers, missing fingers, fused fingers, deformed hand, extra hand, changed gesture, cropped hand, changed pose, illustration, text, watermark"

def prepare_input(path):
    image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    scale = min(1.0, MAX_IMAGE_SIDE / max(image.size))
    width = max(64, int(round(image.width * scale / 64)) * 64)
    height = max(64, int(round(image.height * scale / 64)) * 64)
    return image.resize((width, height), Image.Resampling.LANCZOS)

def seed_for(class_name, filename):
    digest = hashlib.sha256(f"{class_name}/{filename}/artaug-v1".encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big")

def read_manifest_rows():
    with MANIFEST_PATH.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def register_pending_output(class_name, original_filename, output_path):
    relative_name = output_path.relative_to(DATA_DIR).as_posix()
    rows = read_manifest_rows()
    if any(row["filename"] == relative_name for row in rows):
        raise ValueError(f"Manifest already contains {relative_name}; refusing to duplicate a record.")
    fields = ["filename", "class", "source", "original_filename", "split", "approval_status"]
    rows.append({
        "filename": relative_name,
        "class": class_name,
        "source": "artaug",
        "original_filename": original_filename,
        "split": "train",
        "approval_status": "pending",
    })
    temporary = MANIFEST_PATH.with_suffix(".csv.tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temporary, MANIFEST_PATH)

test_pairs = []
for class_name, filename in TEST_INPUTS.items():
    original_path = INPUT_DIR / class_name / filename
    output_path = GENERATED_DIR / class_name / f"{Path(filename).stem}_artaug_v1.png"
    if not original_path.is_file():
        raise FileNotFoundError(f"Test input does not exist: {original_path}")
    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing test output: {output_path}")
    original = prepare_input(original_path)
    try:
        generated = pipe(
            prompt=PROMPTS[class_name],
            negative_prompt=NEGATIVE_PROMPT,
            input_image=original,
            denoising_strength=DENOISING_STRENGTH,
            height=original.height,
            width=original.width,
            seed=seed_for(class_name, filename),
            num_inference_steps=INFERENCE_STEPS,
        )
        if not isinstance(generated, Image.Image):
            raise TypeError(f"DiffSynth returned {type(generated).__name__}, expected a PIL image.")
        generated.save(output_path, format="PNG")
        if output_path.stat().st_size == 0:
            raise IOError(f"Generated output is empty: {output_path}")
        register_pending_output(class_name, filename, output_path)
        test_pairs.append((class_name, original.copy(), generated.copy(), filename, output_path))
        print(f"Saved pending output: {output_path}")
    except Exception as exc:
        raise RuntimeError(f"Test generation failed for {class_name}/{filename}; stop before the remaining batch.") from exc

print(f"Test batch completed: {len(test_pairs)} images. They are still pending manual review.")

## 9. Review original/generated pairs before continuing

Inspect hand visibility, class, exact gesture, finger count, distortion, and composition for every test output. The notebook does not approve any image automatically.

In [ ]:
from IPython.display import display

for class_name, original, generated, filename, output_path in test_pairs:
    print(f"{class_name}: {filename} -> {output_path.name} (AI-generated; pending)")
    display(original, generated)
print("Reject any image with an ambiguous gesture, missing/extra fingers, distorted hand, or major composition change.")

## 10. Generate the remaining 27 images only after review

Run this cell only if all three test pairs preserve their intended gestures. At the prompt, type `TEST_BATCH_PASSED` to continue. This gate authorizes generation only; all resulting manifest records remain `pending` until your later image-by-image review.

In [ ]:
confirmation = input("After manually verifying all 3 test images, type TEST_BATCH_PASSED to generate the other 27: ").strip()
if confirmation != "TEST_BATCH_PASSED":
    raise RuntimeError("Remaining generation was not authorized. The 3 test outputs remain pending.")

test_names = set(TEST_INPUTS.values())
generated_count = 0
for class_name in CLASSES:
    inputs = sorted(p for p in (INPUT_DIR / class_name).iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    if len(inputs) != 10:
        raise ValueError(f"Expected 10 inputs for {class_name}; found {len(inputs)}.")
    for input_path in inputs:
        if input_path.name in test_names:
            continue
        output_path = GENERATED_DIR / class_name / f"{input_path.stem}_artaug_v1.png"
        if output_path.exists():
            raise FileExistsError(f"Refusing to overwrite existing output: {output_path}")
        original = prepare_input(input_path)
        try:
            generated = pipe(
                prompt=PROMPTS[class_name],
                negative_prompt=NEGATIVE_PROMPT,
                input_image=original,
                denoising_strength=DENOISING_STRENGTH,
                height=original.height,
                width=original.width,
                seed=seed_for(class_name, input_path.name),
                num_inference_steps=INFERENCE_STEPS,
            )
            if not isinstance(generated, Image.Image):
                raise TypeError(f"DiffSynth returned {type(generated).__name__}, expected a PIL image.")
            generated.save(output_path, format="PNG")
            if output_path.stat().st_size == 0:
                raise IOError(f"Generated output is empty: {output_path}")
            register_pending_output(class_name, input_path.name, output_path)
            generated_count += 1
            print(f"[{generated_count}/27] Saved pending output: {output_path}")
        except Exception as exc:
            raise RuntimeError(f"Generation stopped at {class_name}/{input_path.name}. Earlier successful files and pending manifest rows were retained.") from exc

print(f"Remaining batch completed: {generated_count} images. No output was approved automatically.")

## 11. Manifest and output verification

This is read-only verification. Generated rows must remain pending. The notebook never changes source rows or approves generated data.

In [ ]:
rows = read_manifest_rows()
generated_rows = [row for row in rows if row["source"] == "artaug"]
for class_name in CLASSES:
    class_rows = [row for row in generated_rows if row["class"] == class_name]
    files = sorted(p for p in (GENERATED_DIR / class_name).iterdir() if p.is_file() and p.suffix.lower() == ".png")
    print(f"{class_name}: {len(files)} generated files; {len(class_rows)} manifest rows")
    for row in class_rows:
        if row["source"] != "artaug" or row["split"] != "train" or row["approval_status"] != "pending":
            raise AssertionError(f"Unexpected generated manifest record: {row}")
        if not (DATA_DIR / row["filename"]).is_file():
            raise FileNotFoundError(f"Manifest output file is missing: {row['filename']}")
if any(row["approval_status"] != "pending" for row in generated_rows):
    raise AssertionError("A generated image is not pending manual approval.")
print(f"Manifest: {MANIFEST_PATH}; total ArtAug rows={len(generated_rows)}")
print("Generated files are separate from the original dataset; validation/test and model artifacts were not accessed.")

## 12. Final summary and limitations

- Output directories: `data/artaug_generated/peace/`, `data/artaug_generated/stop/`, and `data/artaug_generated/thumbs_up/` under the selected Drive `DATA_DIR`.
- Manifest: `data/manifest.csv`; generated records use `source=artaug`, `split=train`, and `approval_status=pending`.
- The source images are unchanged. Generated outputs are not copied into the classifier dataset automatically.
- Manually reject outputs with incorrect class, gesture changes, extra/missing/fused fingers, hidden hands, or major composition changes. Record approval only after visual review.
- This notebook makes no claim that ArtAug improves classification accuracy. Retraining and evaluation are separate experiments using the fixed original validation/test split.
- If the test batch cannot preserve gestures at low denoising strength, stop. The published ArtAug artifact is an aesthetics LoRA rather than a hand-gesture-preserving editor; do not describe failed or unrelated text-to-image outputs as edits of these originals.